# Harness Engineering: a hands-on course

*Lecture notebook. Read the markdown like lecture notes, run the code like a lab. Everything runs offline.*

## The one-line idea
$$\text{Agent} = \text{Model} + \text{Harness}$$

The **model** predicts the next step. The **harness** is everything else: the loop that runs tools, the sandbox, the permissions,
the context the model sees, the verification, and the memory that survives between sessions. Two teams with the *same model*
can ship very different agents. The difference is the harness. Claude Code, Cursor and Codex are harnesses.

## Learning objectives
1. Write the agent loop and say what each line is for.
2. Design tools (schemas, descriptions, error messages) that a model can use well.
3. Enforce **policy** and **sandboxing** in the harness, never in the prompt.
4. Use **verification** (tests) as the agent's feedback signal.
5. Manage **context**: truncation, observation masking, compaction.
6. Hand work across sessions with **progress artifacts** (long-running agents).
7. Read a **trace** to debug an agent.

## The big picture
```
           user task
               |
   +-----------v------------------------------------------+
   |                      HARNESS                          |
   |   context manager --> MODEL --> action                |
   |        ^                           |                  |
   |        |                  policy check (allow/deny)   |
   |        |                           |                  |
   |   observation  <--  tool runs in SANDBOX              |
   |   (truncated)        (read / write / test / shell)    |
   |                                                       |
   |   progress log + git  <-- survives between sessions   |
   +-------------------------------------------------------+
```

## How to use this notebook
- Companion code: `harness.py` (about 200 lines). Open it next to this notebook and read it.
- The "model" is **scripted** so results are deterministic. Section 8 shows how to swap in a real LLM.
- Each section has **Concept**, **Code** and **Checkpoint**.
- **Slide deck** for this course: https://claude.ai/artifact/Me61gX9Et9msRCXVeN8iZR (private to your Claude account)

In [ ]:
import json, textwrap
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from harness import *    # Tool, Sandbox, make_tools, Policy, Context, run_agent, ScriptedBugfixModel, LoopyModel, ...

## 1. The agent loop

**Concept.** An agent is a model running in a loop:

```
repeat up to max_steps:
    action = model(context)             # THINK: the model chooses a tool call or a final answer
    observation = run(action)            # ACT:   the harness executes it (if policy allows)
    context += action, observation       # OBSERVE: the result becomes part of what the model sees next
until the model says it is done
```

The model never touches the world. It only *requests* actions. The harness decides whether and how to run them.
That separation is the whole game: it is where safety, reliability and cost control live.

**Our lab task:** a tiny project has a failing test (`add()` subtracts instead of adding). The agent must fix it.

In [ ]:
ws = make_bugfix_workspace()                 # a temp directory with calc.py (buggy) and test_calc.py
sandbox = Sandbox(ws)
tools = make_tools(sandbox)

print("calc.py before:\n" + (ws / "calc.py").read_text())

result = run_agent(
    model=ScriptedBugfixModel(),             # stand-in for an LLM
    task="Fix the failing test in this project.",
    tools=tools,
    policy=Policy(),
    ctx=Context(),
)
print("\nstatus:", result["status"])
print("calc.py after:\n" + (ws / "calc.py").read_text())

In [ ]:
# The TRACE is the single most important debugging artifact of an agent. Learn to read it.
pd.DataFrame(result["trace"])[["step", "event", "tool", "detail", "tokens"]]

**Reading the trace.** The agent listed files, read `calc.py`, ran tests (failed), wrote a fix, ran tests again (passed), then stopped.
Notice that it did **not** declare success after writing the fix. It ran the tests first.
That *verify before finishing* habit is built into the loop by the harness, not magic in the model.

**Checkpoint 1.**
1. Where in `run_agent` does the model's request become a real action? What sits between the two?
2. What happens if the model asks for a tool that does not exist? (Try it: read the `unknown_tool` branch.)
3. Why does the loop have a `max_steps`?

## 2. Tools: the model's hands

**Concept.** A tool has four parts. The model only ever sees the first three:

| Part | Seen by model? | Purpose |
|---|---|---|
| `name` | yes | how the model refers to it |
| `description` | yes | **the most important prompt you will write**: when to use it, when not to |
| `input_schema` (JSON Schema) | yes | what arguments are valid |
| `fn` | no | the code the harness runs |

**Tool design rules of thumb**
- Name tools by *intent* (`run_tests`), not mechanism (`exec_subprocess`).
- Describe *when* to use a tool and what it returns. Ambiguous descriptions cause wrong tool choices.
- Return **useful errors**: "file not found, did you mean calc.py?" beats a stack trace.
- Keep outputs **small**. A tool that dumps 50 KB teaches the model nothing and burns context.
- Prefer fewer, sharper tools over many overlapping ones. Overlap makes the model hesitate.
- Classify each tool's **risk** (read, write, exec). The harness uses it for policy (section 3).

In [ ]:
# What the model actually receives: names, descriptions and schemas only.
for t in tools.values():
    print(f"{t.name:11} risk={t.risk:9} {t.description}")
print()
print(json.dumps(tools["write_file"].input_schema, indent=2))

**Checkpoint 2.** Rewrite the `read_file` description so a model knows it should call `list_files` first when unsure of a path.
Then ask: could a vague description cause the model to call `run_shell` instead of `run_tests`?

## 3. Policy and sandbox: constraints the model cannot argue with

**Concept.** Never rely on the prompt ("please do not delete files") for safety. A prompt is a *suggestion*.
Policy is *code*: the harness checks every requested action against rules before running it.

Three layers, cheapest first:
1. **Allow-list by risk class**: this policy permits `read`, `write`, `exec_safe`; it denies raw `exec`.
2. **Sandbox**: file tools can only touch one directory. Resolve the path, *then* check containment.
3. **Budgets**: `max_steps` and `max_observation_chars` bound cost and blast radius.

In production add: network egress rules, secrets isolation, container or VM sandboxes, human approval for irreversible actions.

In [ ]:
# A "reckless" model first tries `rm -rf /`. Watch what the harness does.
ws2 = make_bugfix_workspace()
r2 = run_agent(ScriptedBugfixModel(try_dangerous=True), "Fix the failing test.",
               make_tools(Sandbox(ws2)), Policy(), Context())
print("\nfinal status:", r2["status"], "| first event:", r2["trace"][0]["event"])

In [ ]:
# Sandbox escape attempts. The path is resolved BEFORE the containment check, so ".." and symlinks cannot slip through.
for attempt in ["calc.py", "../secrets.txt", "/etc/passwd", "sub/../../outside.txt"]:
    try:
        print(f"{attempt:24} -> OK    {sandbox.resolve(attempt).name}")
    except PermissionError as e:
        print(f"{attempt:24} -> BLOCKED  ({e})")

**Key point.** The reckless request was *denied*, the model received "DENIED..." as an observation, and it carried on and finished the task.
Good harnesses make denials informative so the model can route around them instead of retrying the same forbidden call.

**Checkpoint 3.** Which policy would you change so the agent could run `pytest -x` but not `curl`? (Hint: tools as narrow wrappers beat a general shell plus a deny-list.)

## 4. Verification: feedback is a tool

**Concept.** A model has no ground truth about whether its edit worked. A harness supplies it: run the tests, the linter, the type checker,
a screenshot diff. The agent's reliability is bounded by the *quality of its feedback loop*.

| Feedback signal | Strength | Weakness |
|---|---|---|
| Unit tests | objective, fast | only as good as the tests |
| Type checker / linter | cheap, catches classes of errors | silent on logic bugs |
| End-to-end / browser tests | closest to real behavior | slow, flaky |
| LLM judge ("does this look right?") | covers fuzzy goals | itself unreliable (see the Evals course) |

**A real story from building this notebook.** The first version of `run_tests` reported failure *even after the fix was written*.
The cause: Python had cached the old compiled file (`.pyc`), and the edit kept the same file size within the same second, so the cache looked valid.
The agent was verifying against **stale code**. The fix was one flag (`python -B`). Lesson: **harness bugs are often environment bugs**,
and an agent can look "dumb" when the real fault is its feedback channel.

**Checkpoint 4.** List three ways a test-based feedback loop can mislead an agent (hint: tests that are too weak, tests the agent can edit, caches).
What would stop an agent from "fixing" a failing test by deleting the test?

## 5. Context engineering: what the model gets to see

**Concept.** The model's only knowledge of the task is the **context window** you assemble each step. It is finite, costs money per token,
and quality degrades as it fills with stale material ("context rot"). Context engineering is deciding *what goes in, in what form, and what comes out*.

Techniques, from simplest to most involved:

| Technique | What it does | Trade-off |
|---|---|---|
| **Truncate observations** | cap each tool output | may cut the useful part |
| **Observation masking** | keep the agent's actions, replace old bulky results with a stub | cheap, loses detail |
| **Compaction / summarization** | replace old history with a summary (usually LLM-written) | costs a model call, may drop facts |
| **Note-taking / memory files** | agent writes key facts to a file and re-reads later | needs discipline |
| **Sub-agents** | delegate a subtask to a fresh context, get back only the result | coordination overhead |
| **Just-in-time retrieval** | fetch details when needed instead of preloading | extra tool calls |

**Experiment.** A model reads the same 8 KB file 14 times. Compare token growth with no management, masking and summarization.

In [ ]:
def context_experiment(strategy, budget=6000):
    w = make_bugfix_workspace()
    (w / "big.txt").write_text("x" * 8000)                         # about 2000 tokens per read
    pol = Policy(max_steps=20, max_observation_chars=100_000)       # disable truncation so the effect is visible
    ctx = Context(budget_tokens=budget, strategy=strategy)
    res = run_agent(LoopyModel(), "Study big.txt", make_tools(Sandbox(w)), pol, ctx, verbose=False)
    return [t["tokens"] for t in res["trace"] if "tool" in t], ctx.compactions

fig, ax = plt.subplots(figsize=(8, 4))
for strat in ["none", "mask", "summarize"]:
    series, n = context_experiment(strat)
    ax.plot(range(1, len(series) + 1), series, marker="o", ms=3, label=f"{strat} (compactions: {n})")
ax.axhline(6000, color="gray", ls="--", lw=1, label="budget")
ax.set_xlabel("agent step"); ax.set_ylabel("tokens in context")
ax.set_title("Context growth: unmanaged vs managed"); ax.legend(); plt.show()

**Reading the plot.** With no management the context grows linearly forever (and so does your bill). Masking and summarization pull it back under budget
whenever it crosses the line, producing a sawtooth. Compare what each strategy *kept*:

In [ ]:
for strat in ["mask", "summarize"]:
    w = make_bugfix_workspace(); (w / "big.txt").write_text("x" * 8000)
    ctx = Context(budget_tokens=6000, strategy=strat)
    run_agent(LoopyModel(), "Study big.txt", make_tools(Sandbox(w)), Policy(max_steps=20, max_observation_chars=100_000), ctx, verbose=False)
    print(f"--- {strat}: {len(ctx.messages)} messages left, {ctx.tokens()} tokens")
    for m in ctx.messages[:4]:
        print("   ", m["role"], "|", str(m["content"])[:70].replace("\n", " "))

**Checkpoint 5.**
1. Masking keeps *what the agent did* but drops *what it learned*. When is that fine? When does it hurt?
2. Why summarize only the *old* messages and keep the most recent ones verbatim?
3. A one-line summary like the one we used is lossy. What would you ask an LLM to preserve? (Decisions, file paths touched, open problems.)

## 6. Long-running agents: work in sessions, hand off with artifacts

**Concept.** Real tasks (build a feature, migrate a codebase) outlast one context window. The core problem: **each new session starts with no memory**.
Teams that make long-running agents work typically do **not** run one agent for hours. They run **many short sessions** that hand off through durable artifacts:

- A **feature list** (JSON): every requirement, each with a `passes` flag. Prevents the agent declaring victory early.
- A **progress file**: what was done, what is next, known issues. The first thing a new session reads.
- **Git history**: each session commits, so the next one can see exactly what changed and revert mistakes.
- A **startup routine**: read progress, run the tests, confirm the app still works *before* adding anything new.
- **One feature per session**: incremental progress, no sweeping rewrites.

The simulation below shows the *pattern*, not a real model: each session reads the artifacts, does one unit of work, and updates them.

In [ ]:
import tempfile
proj = Path(tempfile.mkdtemp(prefix="longrun_"))
features = [
    {"id": 1, "description": "user can sign up", "passes": False},
    {"id": 2, "description": "user can log in", "passes": False},
    {"id": 3, "description": "user can reset password", "passes": False},
    {"id": 4, "description": "admin can list users", "passes": False},
]
(proj / "features.json").write_text(json.dumps(features, indent=2))
(proj / "progress.md").write_text("# Progress\n(no sessions yet)\n")

def run_session(n):
    # 1. STARTUP: a fresh session knows nothing, so it reads the artifacts first.
    feats = json.loads((proj / "features.json").read_text())
    notes = (proj / "progress.md").read_text()
    todo = [f for f in feats if not f["passes"]]
    if not todo:
        return f"session {n}: nothing left to do"
    f = todo[0]                                    # 2. ONE feature per session
    # 3. WORK + VERIFY would happen here (the real harness loop from section 1). We just simulate success.
    f["passes"] = True
    (proj / "features.json").write_text(json.dumps(feats, indent=2))
    # 4. HANDOFF: leave a note for the next session (and commit to git in a real project).
    with open(proj / "progress.md", "a") as fh:
        fh.write(f"- session {n}: implemented feature {f['id']} ({f['description']}); tests green. Next: {len(todo) - 1} left.\n")
    return f"session {n}: finished feature {f['id']}"

for n in range(1, 4):
    print(run_session(n))
print()
print((proj / "progress.md").read_text())
print("done flags:", [f["passes"] for f in json.loads((proj / "features.json").read_text())])

**Why a JSON feature list instead of markdown?** Models are less tempted to casually rewrite structured data, and a rule like
"only change the `passes` field" is easy to enforce in the harness.

**Checkpoint 6.** What goes wrong if the progress file says "all done" but features 3 and 4 still fail? Which artifact is the source of truth, and who is allowed to edit it?

## 7. Failure modes and how to debug a harness

| Symptom | Likely harness cause | Fix |
|---|---|---|
| Agent loops forever, repeating one call | no step budget; observation does not change | `max_steps`, detect repeated calls, richer errors |
| Declares success without checking | no verification step in the loop or prompt | make tests a tool, require green before final |
| Calls the wrong tool | overlapping or vague descriptions | sharpen descriptions, remove duplicate tools |
| Forgets earlier decisions | context compaction dropped them | notes file, better summaries |
| Context fills with junk | huge tool outputs | truncate, mask, return summaries |
| Destructive action | safety only in the prompt | policy + sandbox + approvals |
| "Dumb" behavior that looks like a model problem | broken feedback (stale cache, flaky test) | check the harness before blaming the model |

**Debugging method.** Read the **trace** step by step. For each step ask: *what did the model see, what did it ask for, what did the harness do, what came back?*
Most agent bugs are visible in one of those four places.

## 8. Making it real: swap in an LLM

Replace `ScriptedBugfixModel` with a class that has the same `next_action(messages, tools)` method and calls a model API with tool definitions.
Sketch (not executed here, check the current API docs before using):

```python
class LLMModel:
    def __init__(self, client, model_id):
        self.client, self.model_id = client, model_id

    def next_action(self, messages, tools):
        tool_defs = [{"name": t.name, "description": t.description, "input_schema": t.input_schema} for t in tools.values()]
        # 1. convert our `messages` list into the API's message format (including prior tool calls and results)
        # 2. call the model with tool_defs
        # 3. if the response asks for a tool, return {"type": "tool_use", "name": ..., "args": ...}
        #    otherwise return {"type": "final", "text": ...}
```

Everything else (policy, sandbox, truncation, compaction, tracing) stays exactly the same. That is the point of a harness.

## 9. Exercises

1. **Repeated-call detector**: make `run_agent` stop (or warn the model) if it makes the same call with the same args 3 times in a row.
2. **Approval gate**: add a `needs_approval` risk class. Deny it unless a callback returns True. Test with `write_file` on `test_calc.py`.
3. **Protect the tests**: block writes to files matching `test_*.py`. Does the agent still succeed? What if the bug is in the test?
4. **Better summaries**: make `summarize` keep the list of files written and the last test result.
5. **Token-aware truncation**: truncate by estimated tokens, keeping the head and the tail of long output.
6. **A second scenario**: create a workspace with two bugs. Extend `ScriptedBugfixModel` to fix both. Where does scripting become painful? (That pain is why we use LLMs.)
7. **Wire up a real LLM** using section 8, and compare its trace with the scripted one.
8. **Add a sub-agent tool**: `delegate(task)` runs a nested `run_agent` with a fresh `Context` and returns only the final answer.

In [ ]:
# Starter for exercise 1: detect a repeated identical call.
def has_repeated_call(trace, n=3):
    calls = [(t.get("tool"), json.dumps(t.get("args"), sort_keys=True)) for t in trace if "tool" in t]
    return len(calls) >= n and len(set(calls[-n:])) == 1

print(has_repeated_call([{"tool": "read_file", "args": {"path": "a"}}] * 3))   # True
print(has_repeated_call(result["trace"]))                                      # False for our healthy run

## 10. Glossary

| Term | Meaning |
|---|---|
| **Harness** | all the software around a model: loop, tools, policy, context, memory |
| **Agent loop** | think, act, observe, repeat until done |
| **Tool** | a named capability with a schema the model can request |
| **Observation** | the result of a tool call, fed back to the model |
| **Trace** | the recorded sequence of steps; the primary debugging artifact |
| **Policy** | harness-enforced rules about what actions are allowed |
| **Sandbox** | an isolated environment limiting what actions can touch |
| **Context window** | the tokens the model can see this step |
| **Context engineering** | choosing what enters and leaves the context |
| **Compaction** | replacing old history with a summary |
| **Observation masking** | stubbing out old bulky tool outputs |
| **Sub-agent** | a nested agent with its own context, returning only a result |
| **Progress file** | durable notes that let a fresh session resume work |

## 11. Cheat sheet
```
Agent = Model + Harness
Loop      : context -> model -> action -> POLICY -> tool (in SANDBOX) -> observation -> context
Safety    : enforce in code (policy, sandbox, budgets), never only in the prompt
Feedback  : tests/lint/types as tools; agent reliability <= feedback quality
Context   : truncate -> mask -> summarize -> notes -> sub-agents
Long jobs : feature list + progress file + git + one feature per session
Debug     : read the trace; check the harness before blaming the model
```

## 12. Reading list
**Core**
- Anthropic, Effective harnesses for long-running agents: https://anthropic.com/engineering/effective-harnesses-for-long-running-agents
- Anthropic, Building effective agents: https://www.anthropic.com/engineering/building-effective-agents
- Anthropic, Effective context engineering for AI agents: https://www.anthropic.com/engineering/effective-context-engineering-for-ai-agents

**More perspectives**
- What is an AI harness? (Parallel): https://parallel.ai/articles/what-is-an-agent-harness
- What Is an Agent Harness and Why It Matters (Fiddler): https://www.fiddler.ai/blog/what-is-an-agent-harness
- Microsoft Agent Framework, Agent Harness: https://learn.microsoft.com/sr-cyrl-rs/agent-framework/concepts/harness
- Curated list: https://github.com/mahonzhan/awesome-agent-harness

*(The last two Anthropic links were not re-checked when this notebook was written. If one moves, search the title on anthropic.com/engineering.)*

## 13. My experiments log

| # | Change | What happened | What I learned |
|---|--------|---------------|----------------|
| 0 | baseline scripted agent | fixes bug in 6 steps | |
| 1 | | | |
| 2 | | | |

## Where next
1. **Tool use and MCP**: how tools are defined and shared across applications (next course in this series).
2. **Evals**: how to measure whether your harness changes actually help (also in this series).
3. Build a real harness around an LLM and run it on a small repo.